<a href="https://colab.research.google.com/github/michelleasilveira/ST554-Project2/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Creating the Data Check Class

In [4]:
%%writefile SparkDataCheck.py
# Creating Python code from colab
"""
SparkDataCheck.py
-----------------
A data quality class that wraps a Spark SQL DataFrame and provides
methods for validating and summarizing data.

Author: Michelle A Silveira
"""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd


class SparkDataCheck:
    """
    A class that wraps a Spark SQL DataFrame and provides functionality
    for cleaning and checking data quality.

    Attributes
    ----------
    df : pyspark.sql.DataFrame
        The underlying Spark SQL DataFrame.
    """

    # Column types recognized as numeric
    _NUMERIC_TYPES = {
        "float", "int", "long", "bigint", "double", "integer",
        "smallint", "tinyint", "short", "longint"
    }

    def __init__(self, dataframe):
        """
        Initialize SparkDataCheck with a Spark DataFrame.

        Parameters
        ----------
        dataframe : pyspark.sql.DataFrame
            The Spark SQL DataFrame to wrap.
        """
        self.df = dataframe

    @classmethod
    def from_csv(cls, spark, path):
        """
        Create a SparkDataCheck instance by reading a CSV file.

        Parameters
        ----------
        spark : SparkSession
            The active Spark session.
        path : str
            Path to the CSV file (local or HDFS).

        Returns
        -------
        SparkDataCheck
            A new SparkDataCheck instance wrapping the loaded DataFrame.
        """
        df = spark.read.load(
            path,
            format="csv",
            header=True,
            inferSchema=True
        )
        return cls(df)

    @classmethod
    def from_pandas(cls, spark, pandas_df):
        """
        Create a SparkDataCheck instance from a standard pandas DataFrame.

        Parameters
        ----------
        spark : SparkSession
            The active Spark session.
        pandas_df : pandas.DataFrame
            A standard (non-Spark) pandas DataFrame.

        Returns
        -------
        SparkDataCheck
            A new SparkDataCheck instance wrapping the converted DataFrame.
        """
        df = spark.createDataFrame(pandas_df)
        return cls(df)

    # ------------------------------------------------------------------ #
    #  Private helpers                                                     #
    # ------------------------------------------------------------------ #

    def _is_numeric(self, col_name):
        """Return True if the column is a numeric type."""
        col_types = dict(self.df.dtypes)
        if col_name not in col_types:
            return False
        dtype = col_types[col_name].lower()
        # Handle decimal(precision, scale)
        if dtype.startswith("decimal"):
            return True
        return dtype in self._NUMERIC_TYPES

    def _is_string(self, col_name):
        """Return True if the column is a string type."""
        col_types = dict(self.df.dtypes)
        if col_name not in col_types:
            return False
        return col_types[col_name].lower() == "string"

    # ------------------------------------------------------------------ #
    #  Validation methods (return self for chaining)                      #
    # ------------------------------------------------------------------ #

    def check_numeric_bounds(self, col_name, lower=None, upper=None):
        """
        Check whether each value in a numeric column falls within
        user-defined bounds (inclusive).  Appends a boolean column named
        ``<col_name>_in_bounds`` to the DataFrame.  NULL inputs produce
        NULL outputs.

        Parameters
        ----------
        col_name : str
            The numeric column to validate.
        lower : numeric, optional
            Inclusive lower bound.  At least one of lower/upper must be given.
        upper : numeric, optional
            Inclusive upper bound.  At least one of lower/upper must be given.

        Returns
        -------
        self : SparkDataCheck
            Returns itself (chainable).
        """
        if lower is None and upper is None:
            print("Error: At least one of 'lower' or 'upper' must be provided.")
            return self

        if not self._is_numeric(col_name):
            print(
                f"Column '{col_name}' is not numeric "
                f"(type: {dict(self.df.dtypes).get(col_name, 'unknown')}). "
                "No modification made."
            )
            return self

        result_col = f"{col_name}_in_bounds"
        col = F.col(col_name)

        if lower is not None and upper is not None:
            check_expr = F.when(col.isNull(), None).otherwise(col.between(lower, upper))
        elif lower is not None:
            check_expr = F.when(col.isNull(), None).otherwise(col >= lower)
        else:
            check_expr = F.when(col.isNull(), None).otherwise(col <= upper)

        self.df = self.df.withColumn(result_col, check_expr)
        return self

    def check_string_levels(self, col_name, levels):
        """
        Check whether each value in a string column belongs to a set of
        valid levels.  Appends a boolean column named
        ``<col_name>_valid_level`` to the DataFrame.  NULL inputs produce
        NULL outputs.

        Parameters
        ----------
        col_name : str
            The string column to validate.
        levels : list of str
            The allowed values for the column.

        Returns
        -------
        self : SparkDataCheck
            Returns itself (chainable).
        """
        if not self._is_string(col_name):
            print(
                f"Column '{col_name}' is not a string column "
                f"(type: {dict(self.df.dtypes).get(col_name, 'unknown')}). "
                "No modification made."
            )
            return self

        result_col = f"{col_name}_valid_level"
        col = F.col(col_name)

        check_expr = F.when(col.isNull(), None).otherwise(col.isin(levels))
        self.df = self.df.withColumn(result_col, check_expr)
        return self

    def check_missing(self, col_name):
        """
        Check whether each value in a column is NULL.  Appends a boolean
        column named ``<col_name>_is_missing`` to the DataFrame.

        Parameters
        ----------
        col_name : str
            The column to check for missing values.

        Returns
        -------
        self : SparkDataCheck
            Returns itself (chainable).
        """
        result_col = f"{col_name}_is_missing"
        self.df = self.df.withColumn(result_col, F.col(col_name).isNull())
        return self

    # ------------------------------------------------------------------ #
    #  Summarization methods (return a pandas DataFrame)                  #
    # ------------------------------------------------------------------ #

    def numeric_summary(self, col_name=None, group_by=None):
        """
        Report the min and max of a numeric column (or all numeric columns).
        Returns a standard pandas DataFrame.

        Parameters
        ----------
        col_name : str, optional
            The numeric column to summarize.  If None, all numeric columns
            in the DataFrame are summarized.
        group_by : str, optional
            An optional column to group results by.

        Returns
        -------
        pandas.DataFrame or None
            A pandas DataFrame containing min/max values, or None if the
            specified column is not numeric.
        """
        if col_name is not None:
            # --- single column ---
            if not self._is_numeric(col_name):
                print(f"Column '{col_name}' is not numeric.")
                return None

            agg_exprs = [
                F.min(col_name).alias(f"{col_name}_min"),
                F.max(col_name).alias(f"{col_name}_max"),
            ]
            if group_by is not None:
                result = self.df.groupBy(group_by).agg(*agg_exprs).orderBy(group_by)
            else:
                result = self.df.agg(*agg_exprs)

            return result.toPandas()

        else:
            # --- all numeric columns ---
            numeric_cols = [
                c for c, t in self.df.dtypes
                if t.lower() in self._NUMERIC_TYPES or t.lower().startswith("decimal")
            ]

            if not numeric_cols:
                return pd.DataFrame()

            if group_by is not None:
                # Compute per-column and merge on group_by
                partials = []
                for c in numeric_cols:
                    agg_exprs = [
                        F.min(c).alias(f"{c}_min"),
                        F.max(c).alias(f"{c}_max"),
                    ]
                    part = (
                        self.df.groupBy(group_by)
                        .agg(*agg_exprs)
                        .orderBy(group_by)
                        .toPandas()
                    )
                    partials.append(part)
                combined = reduce(
                    lambda left, right: pd.merge(left, right, on=group_by), partials
                )
            else:
                agg_exprs = []
                for c in numeric_cols:
                    agg_exprs.append(F.min(c).alias(f"{c}_min"))
                    agg_exprs.append(F.max(c).alias(f"{c}_max"))
                combined = self.df.agg(*agg_exprs).toPandas()

            return combined

    def string_counts(self, col1, col2=None):
        """
        Report value counts for one or two string columns.
        Returns a standard pandas DataFrame.

        Parameters
        ----------
        col1 : str
            The required string column.
        col2 : str, optional
            An optional second string column for cross-tabulation.

        Returns
        -------
        pandas.DataFrame or None
            A pandas DataFrame of counts, or None if any column is not a string.
        """
        if not self._is_string(col1):
            print(f"Column '{col1}' is not a string column.")
            return None

        if col2 is not None:
            if not self._is_string(col2):
                print(f"Column '{col2}' is not a string column.")
                return None
            group_cols = [col1, col2]
        else:
            group_cols = [col1]

        result = (
            self.df
            .groupBy(*group_cols)
            .count()
            .orderBy(*group_cols)
            .toPandas()
        )
        return result

Overwriting SparkDataCheck.py


Testing Class

In [9]:
!pip install pyspark -q

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd, io, sys, importlib, contextlib

spark = SparkSession.builder.appName("TestSDC").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

import SparkDataCheck as _m; importlib.reload(_m)
from SparkDataCheck import SparkDataCheck

# ── Test fixture ──────────────────────────────────────────────────────────
sdf = spark.createDataFrame(
    [(1, 10.0, "A"), (2, 25.0, "B"), (3, None, "C"), (4, 50.0, None), (5, 100.0, "A")],
    "val INT, score DOUBLE, label STRING"
)

def fresh(): return SparkDataCheck(sdf)
def capture(fn):                          # <-- fixed: uses contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn()
    return buf.getvalue()

results = []

# T1 — both bounds
c = fresh().check_numeric_bounds("score", lower=0, upper=50)
r = {row["score"]: row["score_in_bounds"] for row in c.df.select("score","score_in_bounds").collect()}
results.append(("T1  check_numeric_bounds (both bounds)", r[10.0]==True and r[100.0]==False and r[None] is None))

# T2 — lower only
c = fresh().check_numeric_bounds("score", lower=30)
r = {row["score"]: row["score_in_bounds"] for row in c.df.select("score","score_in_bounds").collect()}
results.append(("T2  check_numeric_bounds (lower only)", r[10.0]==False and r[50.0]==True))

# T3 — upper only
c = fresh().check_numeric_bounds("score", upper=30)
r = {row["score"]: row["score_in_bounds"] for row in c.df.select("score","score_in_bounds").collect()}
results.append(("T3  check_numeric_bounds (upper only)", r[10.0]==True and r[100.0]==False))

# T4 — no bounds → warning
msg = capture(lambda: fresh().check_numeric_bounds("score"))
results.append(("T4  check_numeric_bounds (no bounds → warning)", "least one" in msg))

# T5 — non-numeric column → warning
msg = capture(lambda: fresh().check_numeric_bounds("label", lower=0))
results.append(("T5  check_numeric_bounds (non-numeric → warning)", "not numeric" in msg))

# T6 — string levels
c = fresh().check_string_levels("label", ["A","B"])
r = {row["label"]: row["label_valid_level"] for row in c.df.select("label","label_valid_level").collect()}
results.append(("T6  check_string_levels", r["A"]==True and r["C"]==False and r[None] is None))

# T7 — string levels on numeric column → warning
msg = capture(lambda: fresh().check_string_levels("score", ["x"]))
results.append(("T7  check_string_levels (non-string → warning)", "not a string" in msg))

# T8 — check_missing
c = fresh().check_missing("score")
r = {row["score"]: row["score_is_missing"] for row in c.df.select("score","score_is_missing").collect()}
results.append(("T8  check_missing", r[10.0]==False and r[None]==True))

# T9 — chaining
c = fresh().check_numeric_bounds("score", lower=0, upper=50).check_missing("score").check_string_levels("label",["A"])
results.append(("T9  chaining", all(col in c.df.columns for col in ["score_in_bounds","score_is_missing","label_valid_level"])))

# T10 — numeric_summary single col
res = fresh().numeric_summary(col_name="score")
results.append(("T10 numeric_summary (single col)", float(res["score_min"].iloc[0])==10.0 and float(res["score_max"].iloc[0])==100.0))

# T11 — numeric_summary all cols
res = fresh().numeric_summary()
results.append(("T11 numeric_summary (all cols)", "val_min" in res.columns and "score_min" in res.columns))

# T12 — numeric_summary grouped
res = fresh().numeric_summary(col_name="score", group_by="label")
results.append(("T12 numeric_summary (grouped)", "label" in res.columns and "score_min" in res.columns))

# T13 — numeric_summary non-numeric → None + warning
msg = capture(lambda: fresh().numeric_summary(col_name="label"))
results.append(("T13 numeric_summary (non-numeric → None)", "not numeric" in msg))

# T14 — string_counts single col
res = fresh().string_counts("label")
results.append(("T14 string_counts (single col)", isinstance(res, pd.DataFrame) and "count" in res.columns))

# T15 — string_counts non-string → warning
msg = capture(lambda: fresh().string_counts("val"))
results.append(("T15 string_counts (non-string → warning)", "not a string" in msg))

# T16 — from_pandas
c = SparkDataCheck.from_pandas(spark, pd.DataFrame({"x":[1,2],"y":["a","b"]}))
results.append(("T16 from_pandas", c.df.count()==2 and "x" in c.df.columns))

# T17 — from_csv
import csv
with open("/tmp/mini.csv","w") as f:
    csv.writer(f).writerows([["id","val"],["1","3.5"],["2","7.0"]])
c = SparkDataCheck.from_csv(spark, "/tmp/mini.csv")
results.append(("T17 from_csv", c.df.count()==2 and "val" in c.df.columns))

# ── Results ───────────────────────────────────────────────────────────────
print("\n" + "="*50)
passed = sum(1 for _,ok in results if ok)
for name, ok in results:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
print("="*50)
print(f"  {passed}/{len(results)} tests passed")
print("="*50)